In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os, math, random

#Model link: https://www.kaggle.com/datasets/nallabantuhareeswar/v11-model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


PT_FILE_DIR = "/kaggle/input/datasets/nallabantuhareeswar/v11-model"  
# =================================================================

DATA_PATH = "/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2"
if not os.path.exists(DATA_PATH): DATA_PATH = "/kaggle/input/aisehack-theme-2"

INPUT_STEPS = 10; OUTPUT_STEPS = 16
TARGET = "cpm25"
MET_FEATURES = ["q2", "t2", "u10", "v10", "swdown", "pblh", "psfc", "rain"]
ALL_MONTHS = ["APRIL_16", "JULY_16", "OCT_16", "DEC_16"]
N_FEAT = 11
LOG_PM_MAX = math.log1p(500.0)

# --- PREPROCESSOR STATS ---
class Preprocessor:
    def __init__(self): self.stats = {}
    def compute_stats(self, months):
        base = os.path.join(DATA_PATH, "raw")
        for feat in MET_FEATURES:
            cats = np.concatenate([np.load(os.path.join(base, m, f"{feat}.npy")).astype(np.float32) for m in months], axis=0).ravel()
            valid = cats[~np.isnan(cats)]
            p1, p99 = np.percentile(valid, [1, 99])
            c = np.clip(valid, p1, p99)
            self.stats[feat] = {"mean": float(np.mean(c)), "std": float(np.std(c)) + 1e-6}
        u = np.concatenate([np.load(os.path.join(base, m, "u10.npy")) for m in months])
        v = np.concatenate([np.load(os.path.join(base, m, "v10.npy")) for m in months])
        ws = np.sqrt(u**2 + v**2)
        self.stats["wind_speed"] = {"mean": float(np.mean(ws)), "std": float(np.std(ws)) + 1e-6}
        self.stats["wind_dir"] = {"mean": 0.0, "std": float(np.pi)}
        pm = np.concatenate([np.load(os.path.join(base, m, "cpm25.npy")) for m in months])
        pm_log = np.log1p(np.clip(pm, 0, None))
        self.gw_log_mean = np.mean(pm_log, axis=0)
        self.gw_log_std = np.std(pm_log, axis=0) + 1e-6
        low = self.gw_log_std < 0.1
        self.gw_log_mean[low] = float(np.mean(pm_log)); self.gw_log_std[low] = float(np.std(pm_log)) + 1e-6

print("Computing normalization stats... (Takes 30s)")
prep = Preprocessor(); prep.compute_stats(ALL_MONTHS)
GW_LOG_MEAN_NP = prep.gw_log_mean.copy()
GW_LOG_STD_NP = prep.gw_log_std.copy()

# --- MODEL ARCHITECTURE ---
class TimeChannelUNet(nn.Module):
    def __init__(self, in_steps=10, features=11, out_steps=16, hid=128, drop=0.1):
        super().__init__()
        in_ch = in_steps * features # 110 Channels
        
        self.enc1 = nn.Sequential(nn.Conv2d(in_ch, hid//2, 3, padding=1), nn.SELU(), nn.Dropout2d(drop))
        self.down1 = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(nn.Conv2d(hid//2, hid, 3, padding=1), nn.SELU(), nn.Dropout2d(drop))
        self.down2 = nn.MaxPool2d(2)

        self.neck = nn.Sequential(
            nn.Conv2d(hid, hid*2, 3, padding=1), nn.SELU(),
            nn.Conv2d(hid*2, hid, 3, padding=1), nn.SELU()
        )
        
        self.up1 = nn.ConvTranspose2d(hid, hid//2, 2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(hid + hid//2, hid//2, 3, padding=1), nn.SELU())
        
        self.up2 = nn.ConvTranspose2d(hid//2, hid//4, 2, stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(hid//4 + hid//2, out_steps, 3, padding=1))
        
        self.persistence = nn.Parameter(torch.ones(out_steps))

    def forward(self, x):
        B, T, C, H, W = x.shape
        x_flat = x.view(B, T*C, H, W)
        e1 = self.enc1(x_flat)
        e2 = self.enc2(self.down1(e1))
        neck = self.neck(self.down2(e2))
        d1 = self.dec1(torch.cat([self.up1(neck), e2], dim=1))
        out = self.dec2(torch.cat([self.up2(d1), e1], dim=1))
        last_pm = x[:, -1, 0:1]
        return out + last_pm * self.persistence.view(1, -1, 1, 1)

# --- FAST INFERENCE LOOP ---
test_path = os.path.join(DATA_PATH, "test_in")
# LOAD RAW FEATURES
td = {feat: np.load(os.path.join(test_path, f"{feat}.npy"), mmap_mode="r") for feat in [TARGET] + MET_FEATURES}
nsamples = td["cpm25"].shape[0]
ens_preds = np.zeros((nsamples, 140, 124, OUTPUT_STEPS), dtype=np.float32)

path = os.path.join(PT_FILE_DIR, "v11_model_0.pt")

if os.path.exists(path):
    print(f"✅ Loading Model from {path}...")
    model = TimeChannelUNet(hid=128, drop=0.15).to(device)
    model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    model.eval()
    
    bs = 16
    for s in range(0, nsamples, bs):
        e = min(s + bs, nsamples); cb = e - s
        x_np = np.zeros((cb, INPUT_STEPS, N_FEAT, 140, 124), dtype=np.float32)
        for b, i in enumerate(range(s, e)):
            for t in range(INPUT_STEPS):
                ch = 0
                x_np[b, t, ch] = (np.log1p(np.clip(td[TARGET][i][t].astype(np.float32), 0, None)) - GW_LOG_MEAN_NP) / GW_LOG_STD_NP; ch += 1
                for feat in MET_FEATURES: x_np[b, t, ch] = (td[feat][i][t].astype(np.float32) - prep.stats[feat]["mean"]) / prep.stats[feat]["std"]; ch += 1
                
                u = td["u10"][i][t].astype(np.float32); v = td["v10"][i][t].astype(np.float32)
                x_np[b, t, ch] = (np.sqrt(u**2+v**2) - prep.stats["wind_speed"]["mean"]) / prep.stats["wind_speed"]["std"]; ch += 1
                x_np[b, t, ch] = (np.arctan2(v, u) - prep.stats["wind_dir"]["mean"]) / prep.stats["wind_dir"]["std"]

        with torch.no_grad(), torch.amp.autocast('cuda'):
            pred = model(torch.from_numpy(x_np).to(device)).float()
        
        pred_raw = np.expm1(np.clip(pred.cpu().numpy() * GW_LOG_STD_NP[None, None] + GW_LOG_MEAN_NP[None, None], 0, LOG_PM_MAX))
        ens_preds[s:e] += pred_raw.transpose(0, 2, 3, 1)

    np.save("/kaggle/working/preds.npy", np.clip(ens_preds, 0, 500).astype(np.float32))
    print(f"🎉 INSTANT RECOVERY COMPLETE! Saved 'preds_v11_lightning.npy'.")
else:
    print(f"❌ Could not find {path}. Check your PT_FILE_DIR on Line 11!")


Device: cuda
Computing normalization stats... (Takes 30s)
✅ Loading Model from /kaggle/input/datasets/nallabantuhareeswar/v11-model/v11_model_0.pt...
🎉 INSTANT RECOVERY COMPLETE! Saved 'preds_v11_lightning.npy'.
